# FERQUEUE PCV — Training SSD MobileNet V2 FPNLite 320×320 di Google Colab (`v3`)

Notebook ini dibuat khusus untuk melatih dataset terbaru (`pcv-coco-colab-v3`: **904 train / 387 test**, kelas `Mobil`, `Truk`, `Bus`) di **Google Colab GPU**, lalu mengekspor paket ZIP (`pcv-model-colab-v3.zip`) yang langsung kompatibel dengan perintah `pcv.py import-model` di laptop lokal FERQUEUE.

### Persiapan sebelum menjalankan cell
1. Di menu Google Colab, aktifkan GPU: **Runtime → Change runtime type → T4 GPU** (atau L4 / A100).
2. Upload isi folder lokal `models/mobilenet/runs/pcv-coco-colab-v3/` ke Google Drive di folder:
   ```text
   MyDrive/ferqueue/pcv-coco-colab-v3/
   ├── label_map.pbtxt
   ├── training-plan.json      (opsional, dibaca otomatis jika ada)
   ├── inputs.json             (opsional)
   └── records/
       ├── train.tfrecord      (471 MB — 904 record)
       └── test.tfrecord       (216 MB — 387 record)
   ```
   *(Menaruh `train.tfrecord` dan `test.tfrecord` langsung sejajar dengan `label_map.pbtxt` tanpa subfolder `records/` juga otomatis didukung).*
3. Jalankan setiap cell berurutan dari atas ke bawah. **Checkpoint training disimpan langsung ke Google Drive**, jadi jika Colab terputus di tengah jalan, cukup jalankan ulang Cell 1–4 lalu Cell 6 untuk melanjutkan (*resume*) otomatis dari checkpoint terakhir.


## Cell 1 — Cek GPU, Mount Google Drive, dan Salin TFRecord ke SSD Lokal Colab


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

# 1. Pastikan GPU aktif
gpu_check = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "GPU tidak terdeteksi! Buka menu Runtime -> Change runtime type -> pilih T4 GPU, lalu ulangi Cell 1."
    )
print(gpu_check.stdout)

# 2. Mount Google Drive
drive.mount("/content/drive")

# 3. Path utama (ubah DRIVE_RUN_DIR jika kamu menaruh folder di lokasi Drive lain)
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3")
WORK_DIR = Path("/content/ferqueue_colab")
VENV_DIR = Path("/content/.venv-pcv")
MODEL_DIR = DRIVE_RUN_DIR / "training-unfrozen"   # Folder baru agar tidak resume dari checkpoint lama yang freeze_batchnorm=True
REPORTS_DIR = DRIVE_RUN_DIR / "reports"

WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "records").mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# 4. Cari file input di Drive (mendukung struktur records/*.tfrecord maupun flat)
def find_required(candidates, label):
    for p in candidates:
        if p.is_file():
            return p
    tried = "\n  - ".join(str(p) for p in candidates)
    raise FileNotFoundError(f"File {label} tidak ditemukan. Lokasi yang dicari:\n  - {tried}")

drive_train_tfrec = find_required(
    [DRIVE_RUN_DIR / "records/train.tfrecord", DRIVE_RUN_DIR / "train.tfrecord"],
    "train.tfrecord",
)
drive_test_tfrec = find_required(
    [DRIVE_RUN_DIR / "records/test.tfrecord", DRIVE_RUN_DIR / "test.tfrecord"],
    "test.tfrecord",
)
drive_label_map = find_required(
    [DRIVE_RUN_DIR / "label_map.pbtxt"],
    "label_map.pbtxt",
)

# 5. Salin TFRecord & label_map ke SSD lokal Colab agar I/O training maksimal
LOCAL_TRAIN_TFREC = WORK_DIR / "records/train.tfrecord"
LOCAL_TEST_TFREC = WORK_DIR / "records/test.tfrecord"
LOCAL_LABEL_MAP = WORK_DIR / "label_map.pbtxt"

for src, dst in [
    (drive_train_tfrec, LOCAL_TRAIN_TFREC),
    (drive_test_tfrec, LOCAL_TEST_TFREC),
    (drive_label_map, LOCAL_LABEL_MAP),
]:
    if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
        print(f"Menyalin {src.name} ({src.stat().st_size / (1024**2):.1f} MB) ke SSD lokal Colab...")
        shutil.copy2(src, dst)
    else:
        print(f"Sudah ada di lokal: {dst} ({dst.stat().st_size / (1024**2):.1f} MB)")

# 6. Hyperparameter training (cocok dengan apps/computer-vision/configs/coco-v3-colab.json)
CLASSES = ["Mobil", "Truk", "Bus"]
MODEL_NAME = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
PREPROCESSING = {
    "denoise_kernel": 0,
    "clahe_clip_limit": 0,
    "clahe_grid_size": 8,
    "gamma": 1.0,
}

BATCH_SIZE = 8
NUM_STEPS = 3390        # 30 epoch x 113 steps/epoch (904 gambar train)
LEARNING_RATE = 0.004
FREEZE_BATCHNORM = False  # WAJIB False agar BatchNorm pada box_predictor baru ter-update (mencegah skor kolaps konstan)
CHECKPOINT_EVERY_N = 100
HORIZONTAL_FLIP = True
BRIGHTNESS_DELTA = 0.1

plan_file = DRIVE_RUN_DIR / "training-plan.json"
if plan_file.is_file():
    plan = json.loads(plan_file.read_text(encoding="utf-8"))
    BATCH_SIZE = int(plan.get("batch_size", BATCH_SIZE))
    NUM_STEPS = int(plan.get("num_steps", NUM_STEPS))
    print("Memuat training-plan.json dari Drive:", plan)

print("\nLabel map:")
print(LOCAL_LABEL_MAP.read_text(encoding="utf-8").strip())
print(f"\nSiap! BATCH_SIZE={BATCH_SIZE}, NUM_STEPS={NUM_STEPS}, LR={LEARNING_RATE}, MODEL_DIR={MODEL_DIR}")


## Cell 2 — Pasang Environment Python 3.11 + TensorFlow 2.15.1 (CUDA) Terisolasi

Mengapa memakai `.venv-pcv` (Python 3.11) terpisah?
- TensorFlow Object Detection API membutuhkan **TensorFlow 2.15.1 + Keras 2.15 + Protobuf 3.20.3**, yang hanya tersedia untuk **Python 3.11**.
- Dengan `uv` dan venv terpisah di `/content/.venv-pcv`, instalasi jauh lebih cepat (~1–2 menit), tidak bentrok dengan paket bawaan Colab (Python 3.12 / TF 2.18+), dan **tidak perlu restart runtime**.


In [ ]:
import subprocess
from pathlib import Path

WORK_DIR = Path("/content/ferqueue_colab")
VENV_DIR = Path("/content/.venv-pcv")
VENV_PY = VENV_DIR / "bin/python"

# 1. Pasang uv (manajer Python & pip super cepat)
subprocess.run(["pip", "install", "-q", "uv"], check=True)

# 2. Buat venv Python 3.11 jika belum ada
if not VENV_PY.is_file():
    print("Mengunduh Python 3.11 dan membuat venv /content/.venv-pcv ...")
    subprocess.run(["uv", "python", "install", "3.11"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.11", "--seed", str(VENV_DIR)], check=True)
else:
    print("Venv sudah tersedia:", VENV_PY)

# 3. Tulis requirements yang identik dengan FERQUEUE (training-cuda.txt)
req_file = WORK_DIR / "requirements-colab.txt"
req_file.write_text(
    """Pillow==10.4.0
numpy==1.26.4
opencv-python-headless==4.10.0.84
setuptools==80.9.0
tensorboard==2.15.2
tensorflow[and-cuda]==2.15.1
tf-models-official==2.15.0
tensorflow-text==2.15.0
tensorflow-io==0.36.0
tensorflow-datasets==4.9.3
tensorflow-metadata==1.14.0
keras==2.15.0
tf-keras==2.15.1
protobuf==3.20.3
tf-slim==1.1.0
pycocotools==2.0.7
lvis==0.5.3
lxml==5.3.0
contextlib2==21.6.0
pyparsing==2.4.7
gin-config==0.5.0
PyYAML==6.0.2
six==1.16.0
scipy==1.11.4
matplotlib==3.8.4
pandas==2.1.4
""",
    encoding="utf-8",
)

print("Memasang TensorFlow 2.15.1 (CUDA) & dependensi Object Detection API (tunggu ~1-2 menit)...")
subprocess.run(
    ["uv", "pip", "install", "--python", str(VENV_PY), "-r", str(req_file)],
    check=True,
)
print("Selesai memasang environment Python 3.11 di", VENV_DIR)


## Cell 3 — Unduh & Compile Object Detection API + Patch `tf-slim` + Verifikasi GPU & TFRecord


In [ ]:
import os
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

WORK_DIR = Path("/content/ferqueue_colab")
VENV_DIR = Path("/content/.venv-pcv")
VENV_PY = VENV_DIR / "bin/python"

TF_MODELS_COMMIT = "40695ad8964145a90ae1abbcc740c086e7364eac"
TF_MODELS_URL = f"https://github.com/tensorflow/models/archive/{TF_MODELS_COMMIT}.zip"
TF_MODELS_DIR = WORK_DIR / "tensorflow-models"
PROTOC_URL = "https://github.com/protocolbuffers/protobuf/releases/download/v3.20.3/protoc-3.20.3-linux-x86_64.zip"
PROTOC_DIR = WORK_DIR / "protoc-3.20.3"

# 1. Unduh sumber TensorFlow Models (commit yang dipatok proyek FERQUEUE)
if not (TF_MODELS_DIR / "research/object_detection/protos/pipeline.proto").is_file():
    zip_path = WORK_DIR / "tf-models.zip"
    print("Mengunduh TensorFlow Models API...")
    urllib.request.urlretrieve(TF_MODELS_URL, zip_path)
    extract_tmp = WORK_DIR / "tf-models-extract"
    if extract_tmp.exists():
        shutil.rmtree(extract_tmp)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_tmp)
    extracted_root = extract_tmp / f"models-{TF_MODELS_COMMIT}"
    if TF_MODELS_DIR.exists():
        shutil.rmtree(TF_MODELS_DIR)
    shutil.move(str(extracted_root), str(TF_MODELS_DIR))
    shutil.rmtree(extract_tmp, ignore_errors=True)
    zip_path.unlink(missing_ok=True)
    print("Sumber Object Detection API siap:", TF_MODELS_DIR)
else:
    print("Sumber Object Detection API sudah tersedia:", TF_MODELS_DIR)

# 2. Unduh protoc 3.20.3 dan compile *.proto
protoc_bin = PROTOC_DIR / "bin/protoc"
if not protoc_bin.is_file():
    pzip = WORK_DIR / "protoc.zip"
    print("Mengunduh protoc 3.20.3...")
    urllib.request.urlretrieve(PROTOC_URL, pzip)
    PROTOC_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(pzip) as zf:
        zf.extractall(PROTOC_DIR)
    protoc_bin.chmod(0o755)
    pzip.unlink(missing_ok=True)

research_dir = TF_MODELS_DIR / "research"
if not (research_dir / "object_detection/protos/pipeline_pb2.py").is_file():
    print("Mengompilasi protobuf Object Detection API...")
    protos = sorted(str(p) for p in (research_dir / "object_detection/protos").glob("*.proto"))
    subprocess.run(
        [str(protoc_bin), f"--proto_path={research_dir}", f"--python_out={research_dir}", *protos],
        check=True,
    )
    print("Kompilasi protobuf selesai.")
else:
    print("Protobuf sudah terkompilasi.")

# 3. Helper environment untuk menjalankan skrip di dalam .venv-pcv dengan GPU CUDA 12.2 + OD API
def build_venv_env():
    env = os.environ.copy()
    site_packages = list((VENV_DIR / "lib").glob("python3.*/site-packages"))
    nvidia_libs = [str(p) for sp in site_packages for p in sorted(sp.glob("nvidia/*/lib"))]
    nvidia_bins = [str(p) for sp in site_packages for p in sorted(sp.glob("nvidia/*/bin"))]

    python_paths = [str(research_dir), str(research_dir / "slim")]
    if env.get("PYTHONPATH"):
        python_paths.append(env["PYTHONPATH"])
    env["PYTHONPATH"] = ":".join(python_paths)

    ld_paths = nvidia_libs + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])
    env["LD_LIBRARY_PATH"] = ":".join(ld_paths)

    bin_paths = [str(VENV_DIR / "bin")] + nvidia_bins + ([env["PATH"]] if env.get("PATH") else [])
    env["PATH"] = ":".join(bin_paths)
    env["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    env["TF_CPP_MIN_LOG_LEVEL"] = "1"
    return env

def run_venv(args, stream=True):
    env = build_venv_env()
    cmd = [str(VENV_PY), *args]
    if not stream:
        return subprocess.run(cmd, env=env, check=True, text=True, capture_output=True)
    proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)

# 4. Patch tf-slim decoder (identik dengan pcv.py fix-tf-slim) + cek GPU, SSD forward pass, dan TFRecord
verify_script = WORK_DIR / "verify_env.py"
verify_script.write_text(
    """import importlib.util
from pathlib import Path

# Patch tf-slim decoder
spec = importlib.util.find_spec("tf_slim")
decoder_path = Path(spec.origin).parent / "data/tfexample_decoder.py"
src = decoder_path.read_text(encoding="utf-8")
old_imp = "from tensorflow.python.ops import control_flow_ops"
new_imp = "from tensorflow.compat import v1 as control_flow_ops"
if new_imp not in src and old_imp in src:
    decoder_path.write_text(src.replace(old_imp, new_imp), encoding="utf-8")
    print("Patch tf-slim diterapkan.")
else:
    print("Patch tf-slim sudah sesuai.")

import tensorflow as tf
from tf_slim.data import tfexample_decoder

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__, "| GPU terdeteksi:", gpus)
if not gpus:
    raise RuntimeError("TensorFlow tidak mendeteksi GPU! Periksa runtime GPU Colab.")

# Uji decoder PNG/JPEG dalam tf.function
handler = tfexample_decoder.Image(channels=3)
dummy = tf.zeros([4, 6, 3], dtype=tf.uint8)

@tf.function
def test_decode(encoded, fmt):
    return handler.tensors_to_item({"image/encoded": encoded, "image/format": fmt})

for fmt, enc in ((b"jpeg", tf.io.encode_jpeg(dummy)), (b"png", tf.io.encode_png(dummy))):
    out = test_decode(enc, tf.constant(fmt)).numpy()
    assert out.shape == (4, 6, 3), f"Decoder shape salah: {out.shape}"
print("Decoder tf-slim JPEG/PNG OK.")

# Verifikasi jumlah record TFRecord
for split in ("train", "test"):
    p = Path("/content/ferqueue_colab/records") / f"{split}.tfrecord"
    n = sum(1 for _ in tf.data.TFRecordDataset(str(p)))
    print(f"TFRecord {split}: {n} records")
""",
    encoding="utf-8",
)

run_venv([str(verify_script)])


## Cell 4 — Unduh Pretrained SSD MobileNet V2 FPNLite & Buat `pipeline.config`


In [ ]:
import shutil
import tarfile
import urllib.request
from pathlib import Path

PRETRAINED_DIR = WORK_DIR / "pretrained"
PRETRAINED_MODEL_DIR = PRETRAINED_DIR / MODEL_NAME
PRETRAINED_URL = f"https://storage.googleapis.com/download.tensorflow.org/models/object_detection/tf2/20200711/{MODEL_NAME}.tar.gz"
PIPELINE_CONFIG_PATH = WORK_DIR / "pipeline.config"

# 1. Unduh & ekstrak bobot pretrained COCO resmi
if not (PRETRAINED_MODEL_DIR / "checkpoint/ckpt-0.index").is_file():
    PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
    archive = PRETRAINED_DIR / f"{MODEL_NAME}.tar.gz"
    print(f"Mengunduh pretrained {MODEL_NAME}...")
    urllib.request.urlretrieve(PRETRAINED_URL, archive)
    with tarfile.open(archive, "r:gz") as tf_tar:
        tf_tar.extractall(PRETRAINED_DIR)
    archive.unlink(missing_ok=True)
    print("Pretrained siap:", PRETRAINED_MODEL_DIR)
else:
    print("Pretrained sudah tersedia:", PRETRAINED_MODEL_DIR)

# 2. Generate pipeline.config (identik dengan make_pipeline di ferqueue_cv/training.py)
make_cfg_script = WORK_DIR / "make_pipeline.py"
make_cfg_script.write_text(
    f"""from pathlib import Path
from google.protobuf import text_format
from object_detection.protos import pipeline_pb2, train_pb2

pretrained = Path({str(PRETRAINED_MODEL_DIR)!r})
pipeline = pipeline_pb2.TrainEvalPipelineConfig()
text_format.Parse((pretrained / "pipeline.config").read_text(encoding="utf-8"), pipeline)

pipeline.model.ssd.num_classes = {len(CLASSES)}
pipeline.model.ssd.freeze_batchnorm = {FREEZE_BATCHNORM!r}

train = pipeline.train_config
train.batch_size = {BATCH_SIZE}
train.num_steps = {NUM_STEPS}
train.fine_tune_checkpoint = str(pretrained / "checkpoint/ckpt-0")
train.fine_tune_checkpoint_type = "detection"
train.fine_tune_checkpoint_version = train_pb2.CheckpointVersion.V2
train.sync_replicas = False
train.replicas_to_aggregate = 1

train.optimizer.Clear()
momentum = train.optimizer.momentum_optimizer
momentum.momentum_optimizer_value = 0.9
schedule = momentum.learning_rate.cosine_decay_learning_rate
schedule.learning_rate_base = {LEARNING_RATE}
schedule.total_steps = {NUM_STEPS}
schedule.warmup_steps = {min(100, NUM_STEPS // 10)}
schedule.warmup_learning_rate = {LEARNING_RATE / 10}
train.optimizer.use_moving_average = False

del train.data_augmentation_options[:]
if {HORIZONTAL_FLIP!r}:
    train.data_augmentation_options.add().random_horizontal_flip.SetInParent()
if {BRIGHTNESS_DELTA!r}:
    train.data_augmentation_options.add().random_adjust_brightness.max_delta = {BRIGHTNESS_DELTA}

train.max_number_of_boxes = 100

label_map = {str(LOCAL_LABEL_MAP)!r}
pipeline.train_input_reader.label_map_path = label_map
del pipeline.train_input_reader.tf_record_input_reader.input_path[:]
pipeline.train_input_reader.tf_record_input_reader.input_path.append({str(LOCAL_TRAIN_TFREC)!r})
pipeline.train_input_reader.num_readers = 1

for reader in pipeline.eval_input_reader:
    reader.label_map_path = label_map
    reader.shuffle = False
    reader.num_epochs = 1
    reader.num_readers = 1
    del reader.tf_record_input_reader.input_path[:]
    reader.tf_record_input_reader.input_path.append({str(LOCAL_TEST_TFREC)!r})

pipeline.eval_config.use_moving_averages = False
pipeline.eval_config.batch_size = 1

out = Path({str(PIPELINE_CONFIG_PATH)!r})
out.write_text(text_format.MessageToString(pipeline), encoding="utf-8")
print("pipeline.config dibuat di:", out)
""",
    encoding="utf-8",
)

run_venv([str(make_cfg_script)])
shutil.copy2(PIPELINE_CONFIG_PATH, DRIVE_RUN_DIR / "pipeline.config")
print("Salinan pipeline.config disimpan juga ke:", DRIVE_RUN_DIR / "pipeline.config")


## Cell 5 — (Opsional) Jalankan TensorBoard untuk Pantau Loss
Jalankan cell ini **sebelum** Cell 6 jika ingin melihat grafik `Loss/classification_loss`, `Loss/localization_loss`, dan `Loss/total_loss` secara langsung di Colab. Boleh dilewati jika tidak perlu.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{MODEL_DIR}"


## Cell 6 — Training SSD MobileNet V2 FPNLite di GPU

- Target langkah: **3.390 steps** (30 epoch × 113 steps/epoch, `batch_size = 8`).
- Di GPU T4 Colab, biasanya memakan waktu sekitar **10–20 menit**.
- Checkpoint disimpan tiap **100 steps** ke Google Drive (`MODEL_DIR`). Jika runtime terputus, jalankan ulang Cell 1–4 lalu Cell 6 — training akan **otomatis lanjut dari checkpoint terakhir**.


In [ ]:
print(f"Memulai training hingga {NUM_STEPS} langkah...")
print(f"Checkpoint disimpan di: {MODEL_DIR}")

run_venv([
    "-m", "object_detection.model_main_tf2",
    f"--pipeline_config_path={PIPELINE_CONFIG_PATH}",
    f"--model_dir={MODEL_DIR}",
    f"--num_train_steps={NUM_STEPS}",
    f"--checkpoint_every_n={CHECKPOINT_EVERY_N}",
    "--alsologtostderr",
])

print("\nTraining selesai!")


## Cell 7 — Export SavedModel dari Checkpoint Terakhir


In [ ]:
import json
from pathlib import Path

# Cari checkpoint terakhir di MODEL_DIR (Google Drive)
find_ckpt = run_venv([
    "-c",
    f"import tensorflow as tf; print(tf.train.latest_checkpoint({str(MODEL_DIR)!r}) or '')",
], stream=False)

latest_ckpt = find_ckpt.stdout.strip().splitlines()[-1].strip()
if not latest_ckpt:
    raise RuntimeError(f"Tidak ditemukan checkpoint di {MODEL_DIR}. Pastikan Cell 6 sudah selesai.")

CKPT_NAME = Path(latest_ckpt).name
EXPORT_DIR = WORK_DIR / "exports" / CKPT_NAME
SAVED_MODEL_DIR = EXPORT_DIR / "saved_model"

print(f"Checkpoint terakhir: {CKPT_NAME}")
print(f"Mengekspor SavedModel ke: {EXPORT_DIR} ...")

run_venv([
    "-m", "object_detection.exporter_main_v2",
    "--input_type=image_tensor",
    f"--pipeline_config_path={PIPELINE_CONFIG_PATH}",
    f"--trained_checkpoint_dir={MODEL_DIR}",
    f"--output_directory={EXPORT_DIR}",
])

if not (SAVED_MODEL_DIR / "saved_model.pb").is_file():
    raise RuntimeError("Export gagal: saved_model.pb tidak terbentuk.")

print(f"Export berhasil! SavedModel siap di: {SAVED_MODEL_DIR}")


## Cell 8 — Evaluasi COCO pada `test.tfrecord` (mAP, AP@0.50, AP@0.75, & Per Kelas)

Menghitung metrik COCO identik dengan `pcv.py evaluate --split test` (`ferqueue_cv/coco_metrics.py`) pada **387 gambar test**, lalu menyimpan laporan JSON ke Google Drive (`reports/<ckpt>-test.json`).


In [ ]:
import json
from pathlib import Path

REPORT_PATH = REPORTS_DIR / f"{CKPT_NAME}-test.json"
eval_params_path = WORK_DIR / "eval_params.json"
eval_params_path.write_text(
    json.dumps({
        "classes": CLASSES,
        "saved_model_dir": str(SAVED_MODEL_DIR),
        "test_tfrecord": str(LOCAL_TEST_TFREC),
        "report_path": str(REPORT_PATH),
        "checkpoint": CKPT_NAME,
    }, indent=2),
    encoding="utf-8",
)

eval_script = WORK_DIR / "evaluate_test.py"
eval_script.write_text(
    r"""import copy
import json
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

params = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
CLASSES = params["classes"]
SAVED_MODEL_DIR = Path(params["saved_model_dir"])
TEST_TFREC = Path(params["test_tfrecord"])
REPORT_PATH = Path(params["report_path"])
CKPT_NAME = params["checkpoint"]

SUMMARY_NAMES = (
    "Precision/mAP", "Precision/mAP@.50IOU", "Precision/mAP@.75IOU",
    "Precision/mAP (small)", "Precision/mAP (medium)", "Precision/mAP (large)",
    "Recall/AR@1", "Recall/AR@10", "Recall/AR@100",
    "Recall/AR@100 (small)", "Recall/AR@100 (medium)", "Recall/AR@100 (large)",
)

feature_spec = {
    "image/height": tf.io.FixedLenFeature([], tf.int64),
    "image/width": tf.io.FixedLenFeature([], tf.int64),
    "image/filename": tf.io.FixedLenFeature([], tf.string),
    "image/encoded": tf.io.FixedLenFeature([], tf.string),
    "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
    "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
    "image/object/class/label": tf.io.VarLenFeature(tf.int64),
}

print("Memuat SavedModel:", SAVED_MODEL_DIR)
model = tf.saved_model.load(str(SAVED_MODEL_DIR))
detector = model.signatures["serving_default"]

categories = [{"id": idx, "name": name} for idx, name in enumerate(CLASSES, 1)]
groundtruth = {"info": {}, "images": [], "annotations": [], "categories": categories}
predictions = []

dataset = tf.data.TFRecordDataset(str(TEST_TFREC))
image_count = 0
for image_id, raw_record in enumerate(dataset, 1):
    ex = tf.io.parse_single_example(raw_record, feature_spec)
    height = int(ex["image/height"].numpy())
    width = int(ex["image/width"].numpy())
    filename = ex["image/filename"].numpy().decode("utf-8", errors="replace")
    img = tf.io.decode_image(ex["image/encoded"], channels=3, expand_animations=False)
    img_tensor = tf.cast(img, tf.uint8)[None, ...]

    result = detector(img_tensor)
    count = int(result["num_detections"][0])
    det_boxes = result["detection_boxes"][0, :count].numpy()
    det_classes = result["detection_classes"][0, :count].numpy()
    det_scores = result["detection_scores"][0, :count].numpy()

    groundtruth["images"].append({"id": image_id, "file_name": filename, "width": width, "height": height})

    xmins = tf.sparse.to_dense(ex["image/object/bbox/xmin"]).numpy()
    xmaxs = tf.sparse.to_dense(ex["image/object/bbox/xmax"]).numpy()
    ymins = tf.sparse.to_dense(ex["image/object/bbox/ymin"]).numpy()
    ymaxs = tf.sparse.to_dense(ex["image/object/bbox/ymax"]).numpy()
    labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy()

    for xmin, ymin, xmax, ymax, cid in zip(xmins, ymins, xmaxs, ymaxs, labels):
        x1, y1, x2, y2 = float(xmin * width), float(ymin * height), float(xmax * width), float(ymax * height)
        bw, bh = x2 - x1, y2 - y1
        groundtruth["annotations"].append({
            "id": len(groundtruth["annotations"]) + 1,
            "image_id": image_id,
            "category_id": int(cid),
            "bbox": [x1, y1, bw, bh],
            "area": bw * bh,
            "iscrowd": 0,
        })

    scale = np.array([height, width, height, width], dtype=np.float32)
    for coords, cid, score in zip(det_boxes * scale, det_classes, det_scores):
        ymin, xmin, ymax, xmax = map(float, coords)
        predictions.append({
            "image_id": image_id,
            "category_id": int(cid),
            "bbox": [xmin, ymin, xmax - xmin, ymax - ymin],
            "score": float(score),
        })
    image_count = image_id
    if image_id % 100 == 0:
        print(f"Dievaluasi {image_id} gambar test...")

reference = COCO()
reference.dataset = copy.deepcopy(groundtruth)
reference.createIndex()
if predictions:
    predicted = reference.loadRes(copy.deepcopy(predictions))
else:
    predicted = COCO()
    predicted.dataset = {
        "images": copy.deepcopy(reference.dataset["images"]),
        "categories": copy.deepcopy(reference.dataset["categories"]),
        "annotations": [],
    }
    predicted.createIndex()

evaluator = COCOeval(reference, predicted, iouType="bbox")
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

metrics = {"DetectionBoxes_" + name: float(val) for name, val in zip(SUMMARY_NAMES, evaluator.stats)}
precision = evaluator.eval["precision"]
area_idx = evaluator.params.areaRngLbl.index("all")
max_idx = list(evaluator.params.maxDets).index(100)
names = {cat["id"]: cat["name"] for cat in reference.dataset["categories"]}
for idx, cid in enumerate(evaluator.params.catIds):
    vals = precision[:, :, idx, area_idx, max_idx]
    valid = vals[vals > -1]
    metrics["DetectionBoxes_PerformanceByCategory/mAP/" + names[cid]] = float(np.mean(valid)) if valid.size else -1.0

report = {
    "split": "test",
    "checkpoint": CKPT_NAME,
    "image_count": image_count,
    "metrics": metrics,
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n=== RINGKASAN METRIK TEST ===")
print(f"Checkpoint : {CKPT_NAME} ({image_count} gambar)")
print(f"mAP 50:95  : {metrics['DetectionBoxes_Precision/mAP']:.4f}")
print(f"AP@0.50    : {metrics['DetectionBoxes_Precision/mAP@.50IOU']:.4f}")
print(f"AP@0.75    : {metrics['DetectionBoxes_Precision/mAP@.75IOU']:.4f}")
for cls_name in CLASSES:
    key = f"DetectionBoxes_PerformanceByCategory/mAP/{cls_name}"
    print(f"mAP {cls_name:<6} : {metrics[key]:.4f}")
print("\nLaporan tersimpan di:", REPORT_PATH)
""",
    encoding="utf-8",
)

run_venv([str(eval_script), str(eval_params_path)])


## Cell 9 — Paketkan `pcv-model-colab-v3.zip` Siap Impor ke FERQUEUE

ZIP ini mengikuti format persis `ferqueue_cv/sharing.py` (`model.json` + `saved_model/**`), sehingga bisa langsung diimpor di laptop lokal memakai `pcv.py import-model`.


In [ ]:
import json
import re
import shutil
import zipfile
from pathlib import Path

ARCHIVE_NAME = f"pcv-model-colab-v3-fixed-{CKPT_NAME}.zip"
LOCAL_ARCHIVE = WORK_DIR / ARCHIVE_NAME
DRIVE_ARCHIVE = DRIVE_RUN_DIR / ARCHIVE_NAME

public_meta = {
    "classes": CLASSES,
    "model": MODEL_NAME,
    "checkpoint": CKPT_NAME,
    "preprocessing": PREPROCESSING,
}

# 1. Buat arsip ZIP sesuai format sharing.py
with zipfile.ZipFile(LOCAL_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("model.json", json.dumps(public_meta, ensure_ascii=False, indent=2))
    for path in sorted(SAVED_MODEL_DIR.rglob("*")):
        if path.is_file():
            arcname = "saved_model/" + path.relative_to(SAVED_MODEL_DIR).as_posix()
            zf.write(path, arcname)

# 2. Validasi mandiri agar pasti diterima oleh `pcv.py import-model`
with zipfile.ZipFile(LOCAL_ARCHIVE) as zf:
    names = zf.namelist()
    assert "model.json" in names
    assert "saved_model/saved_model.pb" in names
    assert any(n.startswith("saved_model/variables/") for n in names)
    assert all(n == "model.json" or n.startswith("saved_model/") for n in names)
    meta = json.loads(zf.read("model.json").decode("utf-8"))
    assert re.fullmatch(r"ckpt-\d+", meta["checkpoint"])
    assert meta["model"] == "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"

# 3. Salin ke Google Drive agar mudah diunduh
shutil.copy2(LOCAL_ARCHIVE, DRIVE_ARCHIVE)

size_mb = DRIVE_ARCHIVE.stat().st_size / (1024**2)
print(f"Paket model valid dan tersimpan di Google Drive: {DRIVE_ARCHIVE} ({size_mb:.1f} MB)")
print("\nLangkah impor di laptop lokal FERQUEUE:")
print(f"1. Unduh {ARCHIVE_NAME} dari Google Drive ke folder outputs/{ARCHIVE_NAME}")
print("2. Buat salinan config receiver baru (run_dir kosong), lalu jalankan:")
print(f"   .venv-pcv/bin/python pcv.py import-model --config apps/computer-vision/configs/receiver-v3.json --archive outputs/{ARCHIVE_NAME}")

# Opsional: hapus tanda pagar (#) di bawah jika ingin langsung unduh lewat browser Colab
# from google.colab import files
# files.download(str(LOCAL_ARCHIVE))
